# Demo Modul 8: LSTM dan GRU

Demo ini membandingkan RNN, GRU, dan LSTM pada **anggaran encoder + head yang setara**. Fokusnya bukan mencari satu akurasi tertinggi, melainkan membuktikan bentuk state, menghitung parameter, dan membaca akurasi bersama biaya komputasi.

Gunakan `QUICK_MODE=True` saat sesi 120 menit. Mode penuh disediakan pada notebook starter untuk tugas individual.

In [ ]:
import copy
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

BASE_SEED = 42
QUICK_MODE = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_N, VAL_N, EPOCHS = ((2_400, 600, 2) if QUICK_MODE
                          else (6_000, 1_500, 4))
SEEDS = [BASE_SEED] if QUICK_MODE else [BASE_SEED, BASE_SEED + 1, BASE_SEED + 2]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

seed_everything(BASE_SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'quick_mode': QUICK_MODE,
       'train': TRAIN_N, 'validation': VAL_N, 'epochs': EPOCHS,
       'seeds': SEEDS})

## 1. Retensi informasi pada jalur cell state

Jika tidak ada informasi baru yang ditulis, pengaruh cell state awal setelah $T$ langkah adalah $\prod_t f_t$. Untuk forget gate konstan, nilainya menjadi $f^T$.

In [ ]:
langkah = np.arange(1, 61)
forget_values = [0.5, 0.9, 0.99]

fig, ax = plt.subplots(figsize=(7, 3.5))
for f in forget_values:
    ax.plot(langkah, f ** langkah, label=f'f={f}')
ax.set(xlabel='langkah T', ylabel='retensi $f^T$', yscale='log',
       title='Retensi informasi untuk forget gate konstan')
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

retensi = pd.DataFrame({
    'forget_gate': forget_values,
    'retensi_T20': [f ** 20 for f in forget_values],
    'retensi_T60': [f ** 60 for f in forget_values],
})
print(retensi.to_string(index=False))

**Interpretasi.** Gerbang memberi jalur retensi yang dapat dipelajari, tetapi tidak menjamin gradien tidak pernah lenyap. Bahkan $f=0{,}9$ masih menyusut secara eksponensial.

## 2. Bentuk output, hidden state, dan cell state

In [ ]:
B, T, E, H = 4, 12, 16, 32
x = torch.randn(B, T, E)
panjang = torch.tensor([12, 9, 6, 3])
kemas = pack_padded_sequence(x, panjang.cpu(), batch_first=True,
                               enforce_sorted=False)

for jenis, layer in {
    'rnn': nn.RNN(E, H, batch_first=True),
    'gru': nn.GRU(E, H, batch_first=True),
    'lstm': nn.LSTM(E, H, batch_first=True),
}.items():
    output, state = layer(kemas)
    if jenis == 'lstm':
        h_n, c_n = state
        print(jenis, 'packed.data', tuple(output.data.shape),
              'h_n', tuple(h_n.shape), 'c_n', tuple(c_n.shape))
    else:
        h_n = state
        print(jenis, 'packed.data', tuple(output.data.shape),
              'h_n', tuple(h_n.shape), 'c_n', 'tidak ada')

assert tuple(h_n.shape) == (1, B, H)
assert output.data.shape[0] == int(panjang.sum())

## 3. Parameter: hitung dahulu, baru latih

Satu layer PyTorch memiliki dua bias per kelompok gerbang. Karena itu encoder RNN, GRU, dan LSTM masing-masing memiliki faktor 1, 3, dan 4. Embedding yang sama tidak dipakai saat mencari hidden size, tetapi tetap masuk parameter total.

In [ ]:
EMB, KELAS = 100, 4
FAKTOR = {'rnn': 1, 'gru': 3, 'lstm': 4}

def parameter_teori(jenis: str, hidden: int, emb: int = EMB, kelas: int = KELAS):
    encoder = FAKTOR[jenis] * hidden * (emb + hidden + 2)
    kepala = hidden * kelas + kelas
    return encoder, kepala, encoder + kepala

def parameter_pytorch(jenis: str, hidden: int):
    cls = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}[jenis]
    enc = cls(EMB, hidden, batch_first=True)
    head = nn.Linear(hidden, KELAS)
    return (sum(p.numel() for p in enc.parameters()),
            sum(p.numel() for p in head.parameters()))

baris = []
for jenis in FAKTOR:
    enc_t, head_t, budget_t = parameter_teori(jenis, 96)
    enc_p, head_p = parameter_pytorch(jenis, 96)
    assert (enc_t, head_t) == (enc_p, head_p)
    baris.append({'model': jenis, 'hidden': 96,
                  'encoder': enc_t, 'head': head_t,
                  'encoder_plus_head': budget_t})
print('Hidden size sama (belum adil):')
print(pd.DataFrame(baris).to_string(index=False))

target = parameter_teori('lstm', 96)[2]
def hidden_terdekat(jenis: str, target_budget: int, batas: int = 512):
    return min(range(1, batas + 1),
               key=lambda h: abs(parameter_teori(jenis, h)[2] - target_budget))

KONFIG = {
    'rnn': hidden_terdekat('rnn', target),
    'gru': hidden_terdekat('gru', target),
    'lstm': 96,
}
budget = []
for jenis, hidden in KONFIG.items():
    enc, head, total = parameter_teori(jenis, hidden)
    budget.append({'model': jenis, 'hidden': hidden, 'encoder': enc,
                   'head': head, 'encoder_plus_head': total,
                   'selisih_pct': 100 * (total - target) / target})
tabel_budget = pd.DataFrame(budget)
print('\nAnggaran setara:')
print(tabel_budget.to_string(index=False))
assert tabel_budget['selisih_pct'].abs().max() <= 1.0

**Hasil acuan.** Untuk LSTM $H=96$, pencarian memberikan RNN $H=228$ dan GRU $H=116$. Ketiganya berbeda kurang dari 1% pada encoder + head.

## 4. Pipeline AG News

Vocabulary dibuat hanya dari subset latih. Test set sengaja tidak dibaca karena tidak diperlukan untuk memilih model.

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')
train_file = ROOT / 'train.csv'
if not train_file.exists():
    raise FileNotFoundError(
        f'{train_file} tidak ditemukan. Letakkan AG News CSV sesuai data/README.md')

kolom = ['label', 'judul', 'ringkasan']
data = pd.read_csv(train_file, names=kolom, header=None)
if not str(data.iloc[0]['label']).strip().isdigit():
    data = data.iloc[1:].reset_index(drop=True)
data['teks'] = data['judul'].astype(str) + ' ' + data['ringkasan'].astype(str)
data['y'] = data['label'].astype(int) - 1

idx_train, idx_val = train_test_split(
    np.arange(len(data)), train_size=TRAIN_N, test_size=VAL_N,
    stratify=data['y'].to_numpy(), random_state=BASE_SEED)
teks_train = data['teks'].to_numpy()[idx_train]
teks_val = data['teks'].to_numpy()[idx_val]
y_train = data['y'].to_numpy()[idx_train]
y_val = data['y'].to_numpy()[idx_val]

POLA = re.compile(r"[a-z0-9']+")
def tokenisasi(teks):
    return POLA.findall(str(teks).lower())

cacah = Counter(token for teks in teks_train for token in tokenisasi(teks))
kosakata = ['<pad>', '<unk>'] + [w for w, n in cacah.most_common() if n >= 2]
stoi = {w: i for i, w in enumerate(kosakata)}
V = len(kosakata)
unk = sum(token not in stoi for teks in teks_val for token in tokenisasi(teks))
n_token = sum(len(tokenisasi(teks)) for teks in teks_val)
print({'train': len(idx_train), 'validation': len(idx_val),
       'vocabulary': V, 'validation_unk_pct': 100 * unk / n_token})
assert kosakata[:2] == ['<pad>', '<unk>']
assert sorted(np.unique(y_train)) == [0, 1, 2, 3]

In [ ]:
MAKS = 60
def ke_indeks(daftar_teks):
    X = torch.zeros(len(daftar_teks), MAKS, dtype=torch.long)
    L = torch.zeros(len(daftar_teks), dtype=torch.long)
    for i, teks in enumerate(daftar_teks):
        token = [stoi.get(t, 1) for t in tokenisasi(teks)][:MAKS] or [1]
        X[i, :len(token)] = torch.tensor(token)
        L[i] = len(token)
    return X, L

X_train, L_train = ke_indeks(teks_train)
X_val, L_val = ke_indeks(teks_val)
ds_train = TensorDataset(X_train, L_train, torch.tensor(y_train))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))
print('X train:', tuple(X_train.shape), '| panjang:',
      int(L_train.min()), int(L_train.median()), int(L_train.max()))

## 5. Satu classifier untuk tiga jenis sel

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, jenis: str, hidden: int):
        super().__init__()
        self.jenis, self.hidden = jenis, hidden
        self.embedding = nn.Embedding(V, EMB, padding_idx=0)
        cls = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}[jenis]
        self.encoder = cls(EMB, hidden, batch_first=True)
        self.head = nn.Linear(hidden, KELAS)

    def forward(self, X, panjang):
        emb = self.embedding(X)
        packed = pack_padded_sequence(emb, panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, state = self.encoder(packed)
        h_n = state[0] if self.jenis == 'lstm' else state
        return self.head(h_n[-1])

for jenis, hidden in KONFIG.items():
    model = SequenceClassifier(jenis, hidden)
    encoder_n = sum(p.numel() for p in model.encoder.parameters())
    head_n = sum(p.numel() for p in model.head.parameters())
    assert (encoder_n, head_n) == parameter_pytorch(jenis, hidden)
    with torch.no_grad():
        logits = model(X_train[:4], L_train[:4])
    print(jenis, 'hidden', hidden, 'logits', tuple(logits.shape),
          'encoder+head', encoder_n + head_n)

## 6. Pelatihan terkendali

Norma gradien dicatat **sebelum** clipping. Pengukuran waktu GPU disinkronkan agar tidak hanya mengukur antrean operasi asinkron.

In [ ]:
BATCH = 64

def buat_loader(ds, shuffle: bool, seed: int, batch: int = BATCH):
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=generator)

@torch.no_grad()
def evaluasi(model, ds):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, lb, yb in buat_loader(ds, False, BASE_SEED, 256):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb, lb)
        total_loss += criterion(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

def jalankan(jenis: str, hidden: int, seed: int):
    seed_everything(seed)
    model = SequenceClassifier(jenis, hidden).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    loader = buat_loader(ds_train, True, seed)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'grad_norm': []}
    best_loss, best_state, n_update = float('inf'), None, 0
    sync_device(); mulai = time.perf_counter()

    for _ in range(EPOCHS):
        model.train(); jumlah_loss = 0.0
        for xb, lb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb, lb), yb)
            loss.backward()
            norm = torch.sqrt(sum((p.grad.detach() ** 2).sum()
                                  for p in model.parameters()
                                  if p.grad is not None)).item()
            history['grad_norm'].append(norm)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            jumlah_loss += loss.item() * len(yb); n_update += 1
        val_loss, val_acc = evaluasi(model, ds_val)
        history['train_loss'].append(jumlah_loss / len(ds_train))
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())

    sync_device(); durasi = time.perf_counter() - mulai
    model.load_state_dict(best_state)
    train_loss, _ = evaluasi(model, ds_train)
    val_loss, val_acc = evaluasi(model, ds_val)
    encoder_n = sum(p.numel() for p in model.encoder.parameters())
    head_n = sum(p.numel() for p in model.head.parameters())
    total_n = sum(p.numel() for p in model.parameters())
    catatan = {
        'run_id': f'{jenis}-h{hidden}-s{seed}', 'module': 'M08',
        'seed': seed, 'model': jenis, 'hidden_size': hidden,
        'embedding_size': EMB, 'max_length': MAKS, 'packing': True,
        'train_size': len(ds_train), 'val_size': len(ds_val),
        'epochs': EPOCHS, 'n_updates': n_update,
        'encoder_parameters': encoder_n,
        'encoder_head_parameters': encoder_n + head_n,
        'total_parameters': total_n,
        'model_mib_fp32': 4 * total_n / 1024**2,
        'train_loss': train_loss, 'val_loss': val_loss,
        'val_accuracy': val_acc,
        'grad_norm_mean': float(np.mean(history['grad_norm'])),
        'seconds_per_epoch': durasi / EPOCHS, 'device': str(DEVICE),
        'notes': 'quick demo' if QUICK_MODE else 'full protocol',
    }
    return model, history, catatan


In [ ]:
hasil, riwayat = [], {}
for seed in SEEDS:
    for jenis, hidden in KONFIG.items():
        print(f'Melatih {jenis.upper()} hidden={hidden}, seed={seed} ...')
        _, hist, row = jalankan(jenis, hidden, seed)
        hasil.append(row); riwayat[(jenis, seed)] = hist

tabel = pd.DataFrame(hasil)
kolom_tampil = ['model', 'hidden_size', 'seed', 'encoder_head_parameters',
                 'total_parameters', 'val_loss', 'val_accuracy',
                 'grad_norm_mean', 'seconds_per_epoch']
print(tabel[kolom_tampil].to_string(index=False))
assert tabel.groupby('seed')['n_updates'].nunique().max() == 1
assert len(tabel) == 3 * len(SEEDS)

## 7. Visualisasi dan ringkasan multi-seed

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for jenis in KONFIG:
    hist = riwayat[(jenis, SEEDS[0])]
    epoch = np.arange(1, EPOCHS + 1)
    ax[0].plot(epoch, hist['val_acc'],
               marker='o', label=jenis.upper())
    ax[1].plot(epoch, hist['val_loss'], marker='o', label=jenis.upper())
ax[0].set(xlabel='epoch', ylabel='validation accuracy')
ax[1].set(xlabel='epoch', ylabel='validation loss')
for a in ax:
    a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

ringkasan = (tabel.groupby('model')
             .agg(acc_mean=('val_accuracy', 'mean'),
                  acc_std=('val_accuracy', lambda x: x.std(ddof=0)),
                  seconds_mean=('seconds_per_epoch', 'mean'),
                  encoder_head=('encoder_head_parameters', 'first'),
                  total_parameters=('total_parameters', 'first'),
                  model_mib=('model_mib_fp32', 'first'))
             .reset_index())
print(ringkasan.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(ringkasan['model'].str.upper(), ringkasan['acc_mean'],
       yerr=ringkasan['acc_std'], capsize=5, color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('validation accuracy: mean +/- 1 SD')
ax.grid(axis='y', alpha=.3)
for i, row in ringkasan.iterrows():
    ax.text(i, row['acc_mean'], f"{row['seconds_mean']:.1f} s/epoch",
            ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
output = Path('M08_demo_metrics.csv')
tabel.to_csv(output, index=False)
print(f'{len(tabel)} baris disimpan ke {output}')

## Exit ticket

1. Mengapa hidden size yang sama bukan anggaran yang sama?
2. Apa perbedaan state yang dikembalikan LSTM dan GRU?
3. Model mana yang Anda pilih dari run demo, dan angka apa saja yang mendukung pilihan tersebut?